# 03 — Integration Validation: Population ↔ GeoNames Settlement Joinability

## Purpose

Validate deterministic joinability between government settlements population data and GeoNames settlement records using a reproducible normalization strategy derived from profiling findings.

## Findings

### Source Compatibility Findings

* Government settlement names and GeoNames settlement names are deterministically joinable after normalization.

* One non-settlement government record is present:
  * `KSH code = 0`
  * `LAKCÍM NÉLKÜLI`
  * excluded from integration validation.

### Normalization Findings

* Budapest district naming differs between sources:
  * Government data uses numeric district identifiers
  * GeoNames uses Roman numeral district identifiers

* A normalization rule converting Roman numerals to zero-padded district numbers resolves all Budapest district naming differences.

### GeoNames Findings

* GeoNames contains duplicate settlement names after filtering to populated-place records.

* Duplicate records are infrequent relative to the dataset size.

* Duplicate groups exhibit only minor coordinate variation and do not materially affect settlement-level population analysis.

* A deterministic first-record retention strategy produces a one-to-one settlement mapping.

### Integration Findings

* After normalization and duplicate resolution, every settlement in the government dataset successfully matches a GeoNames settlement record.

* Join coverage is:
  * 3177 / 3177 settlements matched
  * 100.00% coverage

* No unmatched settlements remain after applying the integration rules.

---

## Setup

In [ ]:
import pandas as pd
import re

from hpm.settings import settings

pd.set_option("display.max_colwidth", None)

## Load Governmental Settlement Population Data

In [ ]:
gov = pd.read_excel(
    list(settings.raw_population.iterdir())[-1],
    sheet_name=0,
    header=1,
    index_col=None,
)

gov = gov.rename(columns={"KSH\nkód": "ksh_code", "Település": "ksh_name"})

EXCLUDED_KSH_CODES = {0}  # Corresponds to "LAKCÍM NÉLKÜLI"

gov = gov[~gov["ksh_code"].isin(EXCLUDED_KSH_CODES)].copy()

display(gov)

## Integration Design Decision

GeoNames contains multiple records for certain normalized settlement names.

Inspection of duplicate groups showed that:

- duplicates are rare relative to the full dataset
- coordinate differences are small
- duplicate records do not materially affect settlement-level population analyses

For this project, duplicate GeoNames records were resolved by retaining the first occurrence for each normalized settlement name.

This produces a deterministic one-to-one mapping suitable for settlement-level geospatial enrichment.

## Load GeoNames Settlements

In [ ]:
GEONAMES_COLUMNS = [
    "geonameid","name","asciiname","alternatenames",
    "latitude","longitude","feature class","feature code",
    "county code","cc2","admin1 code","admin2 code",
    "admin3 code","admin4 code","population",
    "elevation","dem","timezone","modification date"
]

geo = pd.read_csv(
    settings.raw_settlements / "HU.txt",
    sep="\t",
    header=None,
    names=GEONAMES_COLUMNS,
)

mask = (
    (geo["name"].notna())
    & (geo["feature code"].isin(["PPL", "PPLA", "PPLA2", "ADM2"]))
)

geo = geo[mask].copy()

geo = geo.drop_duplicates("name", keep="first")

display(geo)

## Normalization Strategy

In [ ]:
ROMAN_TO_INT = {
    "I": 1,
    "II": 2,
    "III": 3,
    "IV": 4,
    "V": 5,
    "VI": 6,
    "VII": 7,
    "VIII": 8,
    "IX": 9,
    "X": 10,
    "XI": 11,
    "XII": 12,
    "XIII": 13,
    "XIV": 14,
    "XV": 15,
    "XVI": 16,
    "XVII": 17,
    "XVIII": 18,
    "XIX": 19,
    "XX": 20,
    "XXI": 21,
    "XXII": 22,
    "XXIII": 23,
}

def normalize(name: str) -> str:
    name = name.strip()

    # Normalize whitespace
    name = re.sub(r"\s+", " ", name)

    # Convert district formats
    match = re.match(r"Budapest ([IVX]+)\.*", name)

    if match:
        roman_dist = match.group(1)
        district = ROMAN_TO_INT[roman_dist]
        return f"Budapest {district:02}"

    return name

## Apply normalization

In [ ]:
gov["norm"] = gov["ksh_name"].apply(normalize)
geo["norm"] = geo["name"].apply(normalize)

## Join validation

In [ ]:
merged = gov.merge(
    geo,
    on="norm",
    how="left",
    indicator=True
)

unmatched = merged[merged["_merge"] != "both"]

## Core assertion

In [ ]:
assert unmatched.empty, (
    f"❌ Unmatched KSH settlements found: {len(unmatched)}"
)

print("✅ All KSH settlements successfully matched to GeoNames")

In [ ]:
total = len(gov)
matched = len(merged[merged["_merge"] == "both"])

print(f"Total KSH settlements: {total}")
print(f"Matched settlements: {matched}")
print(f"Coverage: {matched / total:.2%}")